# Relational databases and first queries (PostgreSQL pilot)

After this block you can explain what tables, rows, columns, data types and keys are, start a PostgreSQL server on your own computer and connect to it from Python, create tables with constraints and load data into them with `COPY`, and write queries that filter, sort and limit rows, including rows with missing values.

This notebook is a **pilot**: the same block as the [SQLite notebook](../01-lecture-relational-databases-and-first-queries.ipynb), on **PostgreSQL** instead of SQLite. It uses the same data (the Berlin snapshot of Inside Airbnb, 26 June 2026, CC BY 4.0, with the tables `listings`, `calendar` and `reviews`) and the same order of sections. Where the concept is the same, the text is short and points to the SQLite notebook; the code shows what PostgreSQL does differently. The last section lists the differences side by side.

> **Note.** The notebook needs two packages that are not yet in the course environment: `pixeltable-pgserver` (PostgreSQL as a Python package) and `psycopg` (the PostgreSQL driver for Python). Start Jupyter with `uv run --with pixeltable-pgserver --with "psycopg[binary]" jupyter lab`, or add both packages to the environment with `uv add pixeltable-pgserver "psycopg[binary]"`.

*Author: course team · Licence: CC BY 4.0, except the sections marked "Adapted from", which keep their source licence (named there).*

## Overview

| Section | What it covers |
|---|---|
| Setup | starting a local PostgreSQL server from Python, building the course database, the helper `q`, a first look at the tables and columns |
| What a relational database is: tables, rows, columns and data types | PostgreSQL's types, real dates and date arithmetic, strict types, `CAST` and `::` |
| Primary keys, foreign keys and relationships | checking the keys of the course data, adding a foreign key to data that break it |
| PostgreSQL: a database server on your computer | servers against files, connection strings, `psql`, the size of the database, read-only sessions |
| Creating tables and loading data: CREATE TABLE, INSERT, COPY | constraints, `INSERT ... RETURNING`, `UPDATE` and `DELETE`, transactions, `COPY`, a strict schema on the real listings |
| Queries: SELECT, WHERE, ORDER BY, LIMIT | the logical order, a `SELECT` alias in `WHERE` and in `ORDER BY`, conditions, `LIKE` and `ILIKE`, `ESCAPE`, regular expressions with `~`, dates, `CASE` |
| NULL values | three-valued logic, `IS NULL`, `COUNT(*)` against `COUNT(col)`, `IS DISTINCT FROM`, the sort order of NULL |
| Worked example | a weekend flat for a family, checked against its calendar |
| SQLite and PostgreSQL side by side | every difference this notebook found |
| Stopping the server | closing the connection and stopping PostgreSQL |

In [ ]:
# Run the code from the repository root, where the case-study paths start.
import os
from pathlib import Path

os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study").is_dir()))

## Setup: start PostgreSQL and open the course database

### Starting a local PostgreSQL server

PostgreSQL is a **database server**: a separate program that holds the data and waits for queries from other programs, its **clients**. Usually an IT department installs and runs it. The package `pixeltable-pgserver` ships the PostgreSQL programs inside a Python package, so `pgserver.get_server(folder)` can start a private server on your own computer. The server keeps its data in the folder you name, here `case-study/data/airbnb/pgdata/`; the first start creates the folder.

Unlike the build script of the SQLite notebook, this server **keeps running in the background**: it is a second program on your computer, and it must run for as long as the notebook sends queries. It stops by itself when the last Python program that uses it ends, for example when you shut down or restart the notebook's kernel (`cleanup_mode="stop"`, the default). The data stay in the folder, and the next start takes about a second. The section on stopping the server at the end shows how to stop it earlier.

In [ ]:
import subprocess
import sys

import pandas as pd
import pixeltable_pgserver as pgserver
import psycopg

PGDATA = Path("case-study/data/airbnb/pgdata")         # the folder where the server keeps its data
server = pgserver.get_server(PGDATA)                    # start the server, or reuse it if it already runs

### Building the course database

The build script [`make_airbnb_postgres.py`](make_airbnb_postgres.py) creates the database `airbnb` on this server and loads the three Parquet files with `COPY` (the section on loading data explains it). It takes a few seconds the first time; afterwards it sees that the tables exist and stops. Like the SQLite notebook, the cell runs the script with `subprocess.run`: the notebook waits until the script ends. The script connects to the server that the previous cell started; it does not start a second one.

In [ ]:
# needs the Airbnb files in case-study/data/airbnb/ (airbnb-v1.zip from the course drive)
subprocess.run([sys.executable, "sessions/03-sql/workbooks/pilot-postgres/make_airbnb_postgres.py"], check=True)

### Connecting from Python

`psycopg.connect(uri)` opens a **connection** to the server. The **connection string** (URI) names the user, the server and the database; `server.get_uri("airbnb")` builds it for the database `airbnb`. Three settings follow:

- `autocommit=True` makes every statement its own transaction. Without it, the first error in a notebook leaves the connection in a failed transaction, and PostgreSQL refuses every later statement with `current transaction is aborted` until you call `con.rollback()`. The section on transactions shows how to group statements on purpose.
- PostgreSQL returns `NUMERIC` values (exact decimals, used for `price`) as Python `Decimal` objects. `register_loader("numeric", FloatLoader)` turns them into ordinary floats, which pandas handles better.
- The helper `q(sql, params)` runs a query and returns the result as a pandas DataFrame, as in the SQLite notebook. `cur.description` holds the names of the result columns.

In [ ]:
from psycopg.types.numeric import FloatLoader

con = psycopg.connect(server.get_uri("airbnb"), autocommit=True)   # a connection to the database airbnb
con.adapters.register_loader("numeric", FloatLoader)                # NUMERIC values arrive as floats


def q(sql, params=None):
    """Run an SQL query on the course database and return the result as a pandas DataFrame."""
    with con.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[column.name for column in cur.description])


q("SELECT version()")

### A first look at the tables and columns

Every PostgreSQL database describes itself in the schema `information_schema`, which is part of the SQL standard (MySQL and Microsoft SQL Server have it too). `information_schema.tables` lists the tables; the course tables lie in the schema `public`, the default place for tables. A **schema** in PostgreSQL is a named group of tables inside a database, like a folder.

**Predict.** How many tables does the course database have?

In [ ]:
q("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' ORDER BY table_name")

`information_schema.columns` has one row per column of every table. `ordinal_position` is the position of the column in the table.

In [ ]:
# find all the columns of the table listings, with their types
q("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'listings'
ORDER BY ordinal_position
""")

Compare the types with the SQLite notebook: `first_review` and `last_review` are `date`, not text, and `price` is `numeric`. The next section explains these types.

## What a relational database is: tables, rows, columns and data types

### Tables, rows and columns

A relational database stores data in tables of rows and columns, each column with a name and a type; the [SQLite notebook](../01-lecture-relational-databases-and-first-queries.ipynb), section "What a relational database is", explains the terms (relation, tuple, attribute, schema, DBMS). They are the same in PostgreSQL.

In [ ]:
q("SELECT id, host_id, district, room_type, accommodates, price, minimum_nights, first_review FROM listings ORDER BY id LIMIT 3")

### Data types in PostgreSQL

PostgreSQL checks every value against the type of its column. The types of the course database:

| Type | Holds | Write a value as | Example in the course data |
|---|---|---|---|
| `INTEGER`, `BIGINT` | a whole number (4 or 8 bytes) | `42` | `accommodates`; `id` needs `BIGINT`, because some ids have 19 digits |
| `NUMERIC(10, 2)` | an exact decimal number, here with 10 digits, 2 after the point | `160.71` | `price` |
| `DOUBLE PRECISION` | a floating-point number (8 bytes) | `52.5` | `latitude`, `longitude` |
| `TEXT` | text of any length | `'Mitte'` | `district`, `room_type` |
| `BOOLEAN` | true or false | `TRUE`, `FALSE` | `available` in `calendar` |
| `DATE` | a day | `DATE '2026-07-01'` | `first_review`, the night in `calendar` |
| `TIMESTAMP`, `INTERVAL` | a day and a time; a length of time | `TIMESTAMP '2026-07-01 14:30'`, `INTERVAL '3 days'` | (not stored) |

`pg_typeof(x)` returns the type of a value. **Predict.** What type does the comparison `price > 100` have?

In [ ]:
q("""
SELECT pg_typeof(id) AS id_type, pg_typeof(price) AS price_type, pg_typeof(first_review) AS first_review_type,
       pg_typeof(price > 100) AS comparison_type, price > 100 AS is_expensive
FROM listings
WHERE price IS NOT NULL
LIMIT 1
""")

A comparison is a `boolean`, and pandas shows it as `True` or `False`, not as 1 or 0 as in SQLite.

### Dates are real dates

Because `first_review` is a `date`, date arithmetic works directly. `DATE '2026-06-26'` writes a date value (a **typed literal**), and the difference of two dates is a whole number of days. In SQLite the same query needs `julianday()` on both sides.

In [ ]:
q("""
SELECT MIN(first_review) AS earliest_first_review,
       MAX(last_review)  AS latest_last_review,
       DATE '2026-06-26' - MIN(first_review) AS days_since_earliest      -- integer days
FROM listings
""")

More date functions, all standard in PostgreSQL:

- `EXTRACT(YEAR FROM d)` returns a part of a date: `YEAR`, `MONTH`, `DAY`, `DOW` (day of the week, 0 = Sunday) or `ISODOW` (1 = Monday to 7 = Sunday);
- `date_trunc('month', d)` cuts a date back to the first day of its month (the result is a timestamp; `::date` makes it a date again, see `CAST` below);
- `d + INTERVAL '1 year'` adds a length of time, and `AGE(a, b)` returns the difference as years, months and days.

In [ ]:
q("""
SELECT id, first_review,
       EXTRACT(YEAR FROM first_review)          AS first_year,
       date_trunc('month', first_review)::date  AS first_month,
       first_review + INTERVAL '1 year'          AS one_year_later,
       AGE(DATE '2026-06-26', first_review)      AS time_on_airbnb
FROM listings
WHERE first_review IS NOT NULL
ORDER BY first_review
LIMIT 3
""")

**Predict.** SQLite stores `'2009-13-01'` without complaint, because it is only text. What does PostgreSQL do with `DATE '2009-13-01'`? The next cell catches the error with `try`/`except`, so that the notebook keeps running.

In [ ]:
try:
    q("SELECT DATE '2009-13-01' AS thirteenth_month")
except psycopg.Error as error:
    print(type(error).__name__, "|", error)

Read the message from the top. `DatetimeFieldOverflow` is the kind of error; `date/time field value out of range: "2009-13-01"` names the value; `LINE 1:` and the `^` point to the place in the query. A real date type rejects an impossible date when it is written, so the strict `CHECK` on dates that the SQLite notebook needs is not necessary here.

### Strict types

**Predict.** A column is declared `INTEGER`. Which of the values `3`, `'4'` and `'three'` does PostgreSQL accept? (In SQLite, an ordinary table accepts all three, and a `STRICT` table accepts the first two.) `CREATE TEMP TABLE` creates a table that disappears when the connection closes.

In [ ]:
con.execute("CREATE TEMP TABLE stay (nights INTEGER)")
for value in ["3", "'4'", "'three'"]:
    try:
        con.execute(f"INSERT INTO stay VALUES ({value})")
        print(f"{value:8} accepted")
    except psycopg.Error as error:
        print(f"{value:8} rejected: {error.diag.message_primary}")
print(con.execute("SELECT nights, pg_typeof(nights) FROM stay").fetchall())

PostgreSQL behaves like a SQLite `STRICT` table: text that is a valid integer is converted, and `'three'` is rejected. `error.diag.message_primary` is the first line of the message, without the `LINE` and `^` part.

### Converting types with `CAST` and `::`

`CAST(expression AS type)` converts a value to another type, as in SQLite. PostgreSQL also accepts the short form `expression::type`. **Predict.** `CAST(160.71 AS INTEGER)` is 160 in SQLite, which cuts off the decimals. Is it the same in PostgreSQL?

In [ ]:
q("""
SELECT price,
       CAST(price AS INTEGER)       AS price_cast,
       price::integer               AS price_short_form,
       TRUNC(price)::integer        AS price_cut_off,
       '12'::integer + 1            AS text_to_number,
       accommodates::text || ' guests' AS guests_text      -- || joins text
FROM listings
WHERE price IS NOT NULL
ORDER BY id
LIMIT 3
""")

PostgreSQL **rounds** when it converts a decimal to an integer: 160.71 becomes 161. `TRUNC(x)` cuts off the decimals, as SQLite's `CAST` does. The same query gives different numbers on the two databases, which is why the side-by-side table at the end lists it.

*`CAST` and column aliases adapted from [Data 100, SQL II](https://github.com/DS-100/fa23-course-notes/blob/main/sql_II/sql_II.qmd) (UC Berkeley Data 100, BSD 3-Clause), here on the Airbnb data.*

## Primary keys, foreign keys and relationships

### Primary keys and foreign keys

A **primary key** identifies each row uniquely and is never empty; a **foreign key** is a column that refers to the primary key of another table. The [SQLite notebook](../01-lecture-relational-databases-and-first-queries.ipynb) explains keys, one-to-many relationships, normalisation and ER diagrams; they are the same in PostgreSQL. The build script declares the primary keys `listings (id)`, `calendar (listing_id, date)` and `reviews (listing_id, month)`, and no foreign keys.

### Checking the keys of the course data

`COUNT(*)` counts rows and `COUNT(DISTINCT column)` counts the different values; if both are equal, no value occurs twice. For the composite keys, the subquery in parentheses keeps the distinct pairs first.

**Predict.** The calendar covers 365 nights per listing. Roughly how many rows does it have?

In [ ]:
q("""
SELECT (SELECT COUNT(*) FROM listings)                                         AS n_listings,
       (SELECT COUNT(DISTINCT id) FROM listings)                               AS n_ids,
       (SELECT COUNT(*) FROM calendar)                                         AS n_calendar_rows,
       (SELECT COUNT(*) FROM (SELECT DISTINCT listing_id, date FROM calendar)) AS n_calendar_keys,
       (SELECT COUNT(*) FROM reviews)                                          AS n_review_rows,
       (SELECT COUNT(*) FROM (SELECT DISTINCT listing_id, month FROM reviews)) AS n_review_keys
""").T.rename(columns={0: "value"})

All three keys hold. Here the check only confirms what the database already guarantees: a declared primary key rejects every duplicate when it is written.

### Adding a foreign key to data that break it

`ALTER TABLE ... ADD CONSTRAINT ... FOREIGN KEY` declares a foreign key on an existing table. PostgreSQL first checks every existing row. **Predict.** Does every `listing_id` in `reviews` refer to an existing listing?

In [ ]:
try:
    con.execute("""
        ALTER TABLE reviews
        ADD CONSTRAINT reviews_listing_fk FOREIGN KEY (listing_id) REFERENCES listings (id)
    """)
except psycopg.errors.ForeignKeyViolation as error:
    print(error)

The message names the constraint, and the `DETAIL` line names the first key that fails: a review for a listing that is not in `listings`. PostgreSQL refuses the constraint, and the table stays as it was. The query below counts all such rows. `WHERE listing_id NOT IN (SELECT id FROM listings)` keeps the rows whose listing is missing (safe here because `id` is never NULL), and `UNION ALL` stacks the rows of two queries with the same columns.

In [ ]:
q("""
SELECT 'calendar' AS child_table, COUNT(*) AS orphan_rows, COUNT(DISTINCT listing_id) AS unknown_listings
FROM calendar WHERE listing_id NOT IN (SELECT id FROM listings)
UNION ALL
SELECT 'reviews', COUNT(*), COUNT(DISTINCT listing_id)
FROM reviews WHERE listing_id NOT IN (SELECT id FROM listings)
""")

Inside Airbnb scrapes listings, calendars and reviews separately, so a listing that went offline between two scrapes appears in one file but not in the other. A foreign key makes you decide what to do with these rows before they are stored.

## PostgreSQL: a database server on your computer

### What a database server is

| | SQLite | PostgreSQL |
|---|---|---|
| Where the data live | one file, opened by your program | a data folder that only the server reads and writes |
| What runs | nothing: the library is part of Python | the server, a separate program, plus your program as client |
| How you connect | the file path | a connection string: user, host, port or socket, database |
| Sharing the data | copy the file | connect to the same server, or export with `pg_dump` |
| Many users writing at once | one writer at a time | yes |

The server in this notebook runs on your own computer and accepts connections only from it. A server of an organisation runs the same program on another machine; only the connection string changes, for example `postgresql://anna@db.example.org:5432/airbnb`.

> **Note.** Do not copy or edit the files in `pgdata/` by hand while the server runs. To pass the database to someone else, give them the build script and the Parquet files, or an export made with `pg_dump`.

### Three ways to connect

1. **From Python** with `psycopg.connect(uri)`, as in this notebook;
2. in a graphical **database client**, such as [DBeaver](https://dbeaver.io/) or [pgAdmin](https://www.pgadmin.org/) (both free), with the same connection details;
3. with PostgreSQL's command-line client `psql`. Commands that start with a backslash, such as `\dt` (list the tables) and `\d reviews` (describe one table), belong to `psql`, not to SQL. `server.psql(commands)` runs `psql` with the given input; `\c airbnb` connects to the course database first.

In [ ]:
print(server.psql("\\c airbnb\n\\dt\n\\d reviews"))

### Looking inside the server

PostgreSQL also describes itself in the schema `pg_catalog`, with functions such as `pg_database_size(name)` and `pg_total_relation_size(table)` (a table with its indexes, in bytes). `pg_size_pretty` writes a size in kB or MB.

**Predict.** Which of the three tables needs the most space, and roughly how many times more than `listings`?

In [ ]:
q("""
SELECT pg_size_pretty(pg_database_size('airbnb'))        AS database,
       pg_size_pretty(pg_total_relation_size('listings')) AS listings,
       pg_size_pretty(pg_total_relation_size('calendar')) AS calendar,
       pg_size_pretty(pg_total_relation_size('reviews'))  AS reviews
""")

The calendar, with 3.8 million rows and an index for its primary key, takes most of the space. The same data in SQLite need about 107 MB: PostgreSQL stores extra information with every row, so that many users can read and write at the same time.

### Read-only sessions

When a program only reads, make the connection read-only, so that a mistake cannot change the data. `SET default_transaction_read_only = on` does this for one connection. On a shared server, the administrator usually gives analysts a user who may only read (`GRANT SELECT`).

In [ ]:
reader = psycopg.connect(server.get_uri("airbnb"), autocommit=True)
reader.execute("SET default_transaction_read_only = on")
try:
    reader.execute("CREATE TABLE should_fail (x INTEGER)")
except psycopg.errors.ReadOnlySqlTransaction as error:
    print("read-only connection refuses to write:", error)
reader.close()

## Creating tables and loading data: CREATE TABLE, INSERT, COPY

### `CREATE TABLE`, constraints and `INSERT`

`CREATE TABLE` defines a table with its columns and rules: `PRIMARY KEY`, `NOT NULL`, `UNIQUE`, `CHECK (condition)`, `DEFAULT value` and `REFERENCES other_table (column)`. The [SQLite notebook](../01-lecture-relational-databases-and-first-queries.ipynb) explains each one. Two differences: PostgreSQL always checks foreign keys (no `PRAGMA foreign_keys = ON`), and it checks the type of every value.

*The list of constraints follows [Data 100, SQL I](https://github.com/DS-100/fa23-course-notes/blob/main/sql_I/sql_I.qmd) (UC Berkeley Data 100, BSD 3-Clause).*

### Creating two small tables with constraints

The demonstration tables go into their own schema `demo`, so that they do not mix with the course tables. `DROP SCHEMA IF EXISTS demo CASCADE` removes an old copy with all its tables, so that the cell can run twice. One `execute` can run several statements separated by semicolons when it has no parameters.

In [ ]:
con.execute("""
DROP SCHEMA IF EXISTS demo CASCADE;
CREATE SCHEMA demo;
CREATE TABLE demo.listings (
    id        INTEGER PRIMARY KEY,                 -- one row per listing
    district  TEXT NOT NULL,
    room_type TEXT NOT NULL CHECK (room_type IN ('Entire home/apt', 'Private room', 'Hotel room', 'Shared room')),
    price     NUMERIC(10, 2) CHECK (price > 0)     -- NULL allowed: no price shown
);
CREATE TABLE demo.reviews (
    listing_id INTEGER NOT NULL REFERENCES demo.listings (id),   -- foreign key, always checked
    month      DATE NOT NULL,
    n_reviews  INTEGER NOT NULL DEFAULT 1 CHECK (n_reviews > 0),
    PRIMARY KEY (listing_id, month)                               -- composite key
);
INSERT INTO demo.listings VALUES (1, 'Mitte', 'Entire home/apt', 120), (2, 'Neukölln', 'Private room', 45),
                                 (3, 'Mitte', 'Entire home/apt', NULL);
INSERT INTO demo.reviews VALUES (1, '2026-04-01', 3), (1, '2026-05-01', 2), (2, '2026-05-01', 1);
""")
q("SELECT * FROM demo.listings")

**Predict.** Which of these five inserts does the database accept, and which rule rejects each of the others? The fifth one has no counterpart in the SQLite notebook.

In [ ]:
invalid = {
    "unknown listing": "INSERT INTO demo.reviews VALUES (9, '2026-05-01', 1)",
    "same month twice": "INSERT INTO demo.reviews VALUES (1, '2026-05-01', 4)",
    "unknown room type": "INSERT INTO demo.listings VALUES (4, 'Mitte', 'Castle', 300)",
    "district missing": "INSERT INTO demo.listings (id, room_type) VALUES (5, 'Private room')",
    "month is not a date": "INSERT INTO demo.reviews VALUES (2, 'May 2026x', 1)",
}
for label, statement in invalid.items():
    try:
        con.execute(statement)
        print(f"{label:20} accepted")
    except psycopg.Error as error:
        print(f"{label:20} rejected ({type(error).__name__}): {error.diag.message_primary}")

Each message names the rule that failed, and the name of the error class says the kind of rule: `ForeignKeyViolation`, `UniqueViolation` (the composite primary key), `CheckViolation`, `NotNullViolation` and `InvalidDatetimeFormat`. PostgreSQL gives each constraint a name, such as `reviews_listing_id_fkey`; `CONSTRAINT name` in `CREATE TABLE` sets your own.

### More ways to change rows: `INSERT ... RETURNING`, `UPDATE` and `DELETE`

`INSERT` with a column list fills the other columns with their `DEFAULT`; `UPDATE ... SET ... WHERE` changes rows and `DELETE FROM ... WHERE` removes them. Without `WHERE`, both act on **every** row. PostgreSQL adds `RETURNING columns`, which returns the rows that the statement wrote, so that you see the result without a second query.

In [ ]:
print(con.execute("INSERT INTO demo.reviews (listing_id, month) VALUES (2, '2026-06-01') RETURNING *").fetchall())
print(con.execute("UPDATE demo.listings SET price = 50 WHERE id = 2 RETURNING id, price").fetchall())
print(con.execute("DELETE FROM demo.reviews WHERE month < '2026-05-01' RETURNING listing_id, month").fetchall())
q("SELECT * FROM demo.reviews ORDER BY listing_id, month")

### Transactions: all or nothing

A **transaction** groups statements so that either all of them are stored or none. `with con.transaction():` opens one: at the end of the block PostgreSQL commits it, and if an error occurs inside the block, it rolls back every statement of the block. **Predict.** The block below inserts a valid listing 4 and then an invalid listing 5. Is listing 4 stored?

In [ ]:
try:
    with con.transaction():
        con.execute("INSERT INTO demo.listings VALUES (4, 'Pankow', 'Private room', 60)")
        con.execute("INSERT INTO demo.listings VALUES (5, 'Pankow', 'Castle', 60)")      # breaks the CHECK
except psycopg.errors.CheckViolation as error:
    print("rolled back:", error.diag.message_primary)
q("SELECT id FROM demo.listings ORDER BY id")

Listing 4 is not stored: the error in the second statement undid the first one. A loading script uses one transaction for the whole load, so that a failed load leaves no half-loaded table.

### `COPY`: loading a file in one statement

`COPY table (columns) FROM ... (FORMAT csv, HEADER)` is PostgreSQL's command for loading many rows at once; it is much faster than one `INSERT` per row. It comes in two forms:

- `COPY ... FROM '/path/file.csv'` reads a file **on the server's computer** and needs special rights;
- `COPY ... FROM STDIN` reads data that the **client sends**. In Python, `cur.copy(...)` opens such a copy and `copy.write(data)` sends the data. In `psql`, `\copy` does the same.

The build script loads the course data this way. Here 200 listings are written to a temporary CSV file (the module `tempfile` keeps it out of the repository) and loaded with `COPY ... FROM STDIN`. An empty field in a CSV file becomes NULL.

In [ ]:
import tempfile

con.execute("CREATE TABLE demo.listings_loaded (id BIGINT PRIMARY KEY, district TEXT NOT NULL, room_type TEXT NOT NULL, price NUMERIC(10, 2))")
sample = q("SELECT id, district, room_type, price FROM listings ORDER BY id LIMIT 200")
csv_path = Path(tempfile.mkdtemp()) / "listings_sample.csv"
sample.to_csv(csv_path, index=False)

with con.cursor() as cur:
    with cur.copy("COPY demo.listings_loaded FROM STDIN (FORMAT csv, HEADER)") as copy:
        copy.write(csv_path.read_text(encoding="utf-8"))
print("rows in file:", len(sample), "| rows in table:", q("SELECT COUNT(*) FROM demo.listings_loaded").iloc[0, 0])

**Predict.** What happens when the same file is loaded a second time?

In [ ]:
try:
    with con.cursor() as cur:
        with cur.copy("COPY demo.listings_loaded FROM STDIN (FORMAT csv, HEADER)") as copy:
            copy.write(csv_path.read_text(encoding="utf-8"))
except psycopg.errors.UniqueViolation as error:
    print("second load refused:", error.diag.message_primary)
    print(error.diag.message_detail)
print("rows in table:", q("SELECT COUNT(*) FROM demo.listings_loaded").iloc[0, 0])

A `COPY` is one statement, so it is stored completely or not at all: the first duplicate id stops it, and the table keeps its 200 rows. The `DETAIL` line names the duplicate key.

### Schema first: what strict rules find in real data

`INSERT INTO table SELECT ...` copies the result of a query into a table. The strict table below adds two rules to the listings: a known room type and a maximum stay below 2,147,483,647, the largest 32-bit integer.

In [ ]:
con.execute("""
CREATE TABLE demo.listings_strict (
    id             BIGINT PRIMARY KEY,
    room_type      TEXT NOT NULL CHECK (room_type IN ('Entire home/apt', 'Private room', 'Hotel room', 'Shared room')),
    minimum_nights INTEGER CHECK (minimum_nights >= 1),
    maximum_nights INTEGER CHECK (maximum_nights < 2147483647),
    CHECK (minimum_nights <= maximum_nights)
)""")
try:
    con.execute("INSERT INTO demo.listings_strict SELECT id, room_type, minimum_nights, maximum_nights FROM listings")
except psycopg.errors.CheckViolation as error:
    print("rejected:", error.diag.message_primary)
    print(error.diag.message_detail[:80], "...")
print("listings stored:", q("SELECT COUNT(*) FROM demo.listings_strict").iloc[0, 0])

The database refuses the whole statement, and the message names the constraint that failed; the `DETAIL` line shows the first row that broke it. Which rows break the rule?

In [ ]:
q("SELECT id, minimum_nights, maximum_nights FROM listings WHERE maximum_nights >= 2147483647")

Two listings carry the maximum stay 2,147,483,647: a software default, not a stay. The loader must now decide what to do with them and write the decision down. The demonstration tables are no longer needed:

In [ ]:
con.execute("DROP SCHEMA demo CASCADE")      # removes the schema demo with all its tables

## Queries: SELECT, WHERE, ORDER BY, LIMIT

### The basic clauses

The clauses and their written order are the same as in SQLite:

```
SELECT <column list>
FROM <table>
[WHERE <condition>]
[ORDER BY <column list>]
[LIMIT <number of rows>]
[OFFSET <number of rows>];
```

PostgreSQL also accepts the standard form `FETCH FIRST 5 ROWS ONLY` for `LIMIT 5`.

*The clause template adapted from [Data 100, SQL I](https://github.com/DS-100/fa23-course-notes/blob/main/sql_I/sql_I.qmd) (UC Berkeley Data 100, BSD 3-Clause).*

### The logical order of the clauses

```mermaid
flowchart LR
    F["1 FROM"] --> W["2 WHERE"] --> G["3 GROUP BY"] --> H["4 HAVING"] --> S["5 SELECT"] --> O["6 ORDER BY"] --> L["7 LIMIT / OFFSET"]
```

#### A `SELECT` alias in `WHERE` and in `ORDER BY`

An alias is created in step 5, `SELECT`. When `WHERE` runs in step 2, the alias does not exist yet. PostgreSQL follows the standard here. **Predict.** Which line does the error message point to?

In [ ]:
try:
    q("""
    SELECT id, price, ROUND(price / accommodates, 1) AS price_per_guest
    FROM listings
    WHERE price_per_guest > 50
    """)
except psycopg.errors.UndefinedColumn as error:
    print(error)

`column "price_per_guest" does not exist` names the alias, and `LINE 4:` with the `^` points to the `WHERE` line. (SQLite runs the same query without an error; exercise 1.4 of the SQLite notebook asks about it.) There are two fixes: repeat the expression in `WHERE`, or compute the column in a subquery first (block 2). In `ORDER BY`, step 6, the alias exists:

In [ ]:
q("""
SELECT id, price, accommodates, ROUND(price / accommodates, 1) AS price_per_guest
FROM listings
WHERE price / accommodates > 50                -- fix: repeat the expression
ORDER BY price_per_guest DESC                  -- the alias exists here: ORDER BY runs after SELECT
LIMIT 3
""")

> **Note.** `ROUND(x, 1)` works here because `price` is `NUMERIC`. For a `DOUBLE PRECISION` column such as `latitude`, PostgreSQL has only `ROUND(x)` without digits; write `ROUND(latitude::numeric, 2)`.

### SELECT: columns, aliases, expressions and DISTINCT

`SELECT DISTINCT` removes duplicate rows from the result, as in SQLite.

In [ ]:
q("SELECT DISTINCT room_type FROM listings ORDER BY room_type")

### WHERE: conditions

Conditions use `=`, `<>`, `<`, `<=`, `>`, `>=`, combined with `AND`, `OR` and `NOT`. Text is written in single quotes; double quotes are for names of columns and tables only, and PostgreSQL treats `"Mitte"` as a column name.

#### `AND`, `OR` and parentheses

**Predict.** `AND` binds more strongly than `OR`. Which count answers "private rooms in Mitte or Pankow"?

In [ ]:
q("""
SELECT
  (SELECT COUNT(*) FROM listings
   WHERE room_type = 'Private room' AND district = 'Mitte' OR district = 'Pankow')   AS without_parentheses,
  (SELECT COUNT(*) FROM listings
   WHERE room_type = 'Private room' AND (district = 'Mitte' OR district = 'Pankow')) AS with_parentheses
""")

#### Shorter conditions: `IN`, `BETWEEN` and `NOT`

In [ ]:
q("""
SELECT district, room_type, price
FROM listings
WHERE district IN ('Mitte', 'Pankow') AND room_type NOT IN ('Entire home/apt')
  AND price BETWEEN 50 AND 60              -- same as price >= 50 AND price <= 60
ORDER BY price, id
LIMIT 5
""")

#### Text patterns with `LIKE` and `ILIKE`

In a `LIKE` pattern, `%` stands for any number of characters and `_` for exactly one. In PostgreSQL, `LIKE` is **case-sensitive**; `ILIKE` ignores case. **Predict.** SQLite's `LIKE '%balcony%'` finds 412 titles. How many do `LIKE` and `ILIKE` find in PostgreSQL?

A condition is a `boolean`, so `COUNT(*) FILTER (WHERE condition)` counts the rows where it is true. (`SUM(condition)`, as in the SQLite notebook, fails here: PostgreSQL has no sum of booleans.)

In [ ]:
q("""
SELECT COUNT(*) FILTER (WHERE name LIKE '%balcony%')                          AS like_lower_case,
       COUNT(*) FILTER (WHERE name ILIKE '%balcony%')                         AS ilike_any_case,
       COUNT(*) FILTER (WHERE name ILIKE '%balcony%' OR name ILIKE '%balkon%') AS ilike_balcony_or_balkon
FROM listings
""")

`LIKE` finds only the titles with a lower-case "balcony". `ILIKE` finds all 412, and with "balkon" it finds 583, the same numbers as SQLite's `LIKE`. Unlike SQLite, `ILIKE` also ignores the case of letters such as `ö`:

In [ ]:
q("""
SELECT COUNT(*) FILTER (WHERE district ILIKE 'NEUKÖLLN') AS ilike_capital_umlaut,
       COUNT(*) FILTER (WHERE district LIKE 'NEUKÖLLN')  AS like_capital_umlaut,
       upper('neukölln')                                AS upper_case
FROM listings
""")

`_` matches exactly one character. **Predict.** Which neighbourhoods does `'West _'` match: "West 1" to "West 5", "Westend", or both?

In [ ]:
q("SELECT DISTINCT neighbourhood FROM listings WHERE neighbourhood LIKE 'West _' ORDER BY neighbourhood")

*`LIKE` with `%` and `_` adapted from [Data 100, SQL II](https://github.com/DS-100/fa23-course-notes/blob/main/sql_II/sql_II.qmd) (UC Berkeley Data 100, BSD 3-Clause), here on the Airbnb data.*

#### `NOT LIKE` and `ESCAPE`

`NOT LIKE` and `NOT ILIKE` keep the rows that do not match. To search for `%` or `_` themselves, put the escape character in front of them. In PostgreSQL the escape character is the backslash `\` by default; `ESCAPE '!'` sets another one. In Python, write such a query as a raw string `r"""..."""`, so that Python keeps the backslash.

**Predict.** `name LIKE '%50%%'` is meant to find titles that say "50%". How many does it find?

In [ ]:
q(r"""
SELECT COUNT(*) FILTER (WHERE name LIKE '%50%%')                       AS percent_as_wildcard,
       COUNT(*) FILTER (WHERE name LIKE '%50\%%')                      AS literal_50_percent,
       COUNT(*) FILTER (WHERE name LIKE '%!%%' ESCAPE '!')             AS any_percent_sign,
       COUNT(*) FILTER (WHERE name NOT ILIKE '%balcony%' AND name NOT ILIKE '%balkon%') AS without_balcony
FROM listings
""")

The first pattern finds 46 titles that contain "50" somewhere, such as "50 m²" or "150 qm": the middle `%` is a wildcard. With the escape character, no title says "50%", and two titles contain a percent sign at all. 9,638 titles mention no balcony; 583 + 9,638 = 10,221, all listings.

#### Regular expressions with `~`

PostgreSQL has regular expressions built in, with no extra function as in SQLite:

| Operator | Means |
|---|---|
| `text ~ 'pattern'` | the pattern occurs, case-sensitive |
| `text ~* 'pattern'` | the pattern occurs, ignoring case |
| `text !~ 'pattern'`, `text !~* 'pattern'` | the pattern does not occur |
| `regexp_replace(text, 'pattern', 'new')` | replaces the matches |

The syntax is close to Python's `re`, with one trap: a word boundary is `\y` in PostgreSQL, while `\b` means the backspace character. The pattern `\y[0-9]+ ?(sqm|qm|m²)` reads: a word boundary, one or more digits, an optional space, and one of three units.

In [ ]:
q(r"""
SELECT COUNT(*) FILTER (WHERE name ~ '\y[0-9]+ ?(sqm|qm|m²)')  AS titles_with_size,
       COUNT(*) FILTER (WHERE name ~ '\b[0-9]+ ?(sqm|qm|m²)')  AS with_python_boundary,
       COUNT(*) FILTER (WHERE name ~* 'balcony|balkon')         AS with_balcony
FROM listings
""")

With `\b` the pattern finds nothing, and PostgreSQL gives no warning: a pattern copied from Python must be checked. `~*` finds the same 583 balcony titles as the two `ILIKE` conditions. The standard SQL operator `SIMILAR TO` also exists, but it mixes `LIKE` and regular expressions, and it is rarely used.

#### Conditions on dates

`calendar.date` is a `date`, so a range is written with date literals, and `EXTRACT(DOW FROM date)` returns the weekday as a number (0 is Sunday, 6 Saturday). **Predict.** `available` is a `boolean`. Does `AVG(available)` give the share of available nights, as in SQLite?

In [ ]:
try:
    q("SELECT AVG(available) FROM calendar")
except psycopg.errors.UndefinedFunction as error:
    print(error.diag.message_primary, "|", error.diag.message_hint)

PostgreSQL has no average of true/false values, and the hint suggests a cast: `available::int` turns `true` into 1 and `false` into 0. The share of Saturday nights in August that are still available:

In [ ]:
q("""
SELECT COUNT(*) AS saturday_nights, ROUND(AVG(available::int), 3) AS share_available
FROM calendar
WHERE date BETWEEN DATE '2026-08-01' AND DATE '2026-08-31' AND EXTRACT(DOW FROM date) = 6
""")

The same numbers as in SQLite. A night that is not available is booked or blocked by the host; the calendar does not say which.

### ORDER BY, LIMIT and OFFSET

`ORDER BY a, b` sorts by `a` and breaks ties with `b`; `LIMIT 3 OFFSET 3` returns rows 4 to 6.

In [ ]:
q("""
SELECT id, district, room_type, minimum_nights
FROM listings
ORDER BY minimum_nights DESC NULLS LAST, id
LIMIT 3 OFFSET 0       -- try OFFSET 3 for the next page
""")

`NULLS LAST` is needed here: in descending order, PostgreSQL puts NULL **first**, before every number. The section on NULL values explains the sort order.

> **Note.** Without `ORDER BY`, the order of rows is not defined, in PostgreSQL as in SQLite.

### CASE: a value that depends on a condition

`CASE WHEN condition THEN value ... ELSE value END` works as in SQLite.

In [ ]:
q("""
SELECT id, minimum_nights,
       CASE WHEN minimum_nights < 7  THEN 'short stay'
            WHEN minimum_nights < 28 THEN 'week or more'
            ELSE 'month or more'
       END AS stay_type
FROM listings
ORDER BY id
LIMIT 5
""")

*`CASE` adapted from [Data 100, SQL II](https://github.com/DS-100/fa23-course-notes/blob/main/sql_II/sql_II.qmd) (UC Berkeley Data 100, BSD 3-Clause).*

> 📖 **Read more:** [PostgreSQL tutorial](https://www.postgresql.org/docs/current/tutorial-sql.html) (PostgreSQL Global Development Group): tables, queries and joins in the official documentation, chapter 2.

## NULL values

### NULL and three-valued logic

NULL means "missing" or "unknown", and SQL treats it with three-valued logic: a condition is true, false or unknown. The [SQLite notebook](../01-lecture-relational-databases-and-first-queries.ipynb) explains the rules; they are the same in PostgreSQL, with two differences:

- `a IS b` in SQLite becomes `a IS NOT DISTINCT FROM b` in PostgreSQL: it compares like `=`, but treats two NULLs as equal. `a IS DISTINCT FROM b` is its opposite.
- PostgreSQL sorts NULL **after** every other value: last in ascending order, first in descending order. SQLite does the opposite.

### NULL in conditions and counts

**Predict.** Do the first two counts add up to all listings?

In [ ]:
q("""
SELECT (SELECT COUNT(*) FROM listings WHERE price > 150)          AS above_150,
       (SELECT COUNT(*) FROM listings WHERE NOT (price > 150))    AS not_above_150,
       (SELECT COUNT(*) FROM listings WHERE price IS NULL)        AS no_price,
       (SELECT COUNT(*) FROM listings WHERE price = NULL)         AS equals_null,
       (SELECT COUNT(*) FROM listings)                            AS all_listings
""")

Only the first three numbers together give all listings, as in SQLite: `price = NULL` is never true.

### NULL in expressions, `IS DISTINCT FROM` and the sort order

**Predict.** In ascending order of price, do the listings without a price come first or last?

In [ ]:
q("""
SELECT id, price, price * 2 AS double_price, COALESCE(price, -1) AS price_or_minus_one,
       price IS DISTINCT FROM 120 AS differs_from_120
FROM listings
WHERE district = 'Mitte'
ORDER BY price DESC          -- descending: PostgreSQL puts the NULLs first; try NULLS LAST
LIMIT 3
""")

In descending order the listings without a price come first, and `price * 2` stays NULL. `price IS DISTINCT FROM 120` is `True` for a missing price, while `price <> 120` would be NULL. Write `NULLS FIRST` or `NULLS LAST` whenever a column can be NULL and the order matters, so that the query gives the same order on every database.

### Is a missing price random?

In [ ]:
q("""
SELECT COUNT(*) FILTER (WHERE price IS NULL)                              AS no_price,
       COUNT(*) FILTER (WHERE price IS NULL AND availability_365 = 0)     AS no_price_no_free_night,
       COUNT(*) FILTER (WHERE price IS NOT NULL AND availability_365 = 0) AS price_no_free_night
FROM listings
""")

Four in five listings without a price have no free night at all: they are probably not offered at the moment. `COUNT(*) FILTER (WHERE ...)` gives the three counts in one pass over the table, instead of three subqueries.

## Worked example · Which flat can a family book for a weekend?

A family of four looks for an entire flat in Friedrichshain-Kreuzberg for a weekend: a minimum stay of at most two nights, at least four guests, at most €150 a night, and at least 20 reviews.

### Step 1 · Turn the family's wishes into conditions and a sort order

**Why this step?** Why sort by rating and then by number of reviews, and not by rating alone?

In [ ]:
family = q("""
SELECT id, neighbourhood, accommodates, price, number_of_reviews, review_scores_rating
FROM listings
WHERE district = 'Friedrichshain-Kreuzberg'
  AND room_type = 'Entire home/apt'
  AND minimum_nights <= 2 AND accommodates >= 4 AND price <= 150 AND number_of_reviews >= 20
ORDER BY review_scores_rating DESC NULLS LAST, number_of_reviews DESC
LIMIT 5
""")
family

### Step 2 · Check the calendar of the best flat

The id goes into the query as a **parameter**: psycopg writes `%s` where SQLite writes `?`, and passes the values separately (block 2 explains why this is safer than pasting the id into the text). `to_char(date, 'Dy')` writes the weekday as a short name.

> **Note.** In a query with parameters, psycopg reads every `%` as the start of a placeholder; write `%%` for a `LIKE` wildcard.

**Why this step?** The flat passed every condition of step 1. Why look at a second table at all?

In [ ]:
best = int(family.loc[0, "id"])
q("""
SELECT date, to_char(date, 'Dy') AS weekday, available
FROM calendar
WHERE listing_id = %s AND date BETWEEN DATE '2026-07-31' AND DATE '2026-08-02'
ORDER BY date
""", params=(best,))

### Step 3 · Read the answer and decide

If the best-rated flat is booked for that weekend, the family moves down the list of step 1 and checks the next flat's calendar in the same way. Block 2 combines both steps into one query with a join.

## SQLite and PostgreSQL side by side

Every difference this notebook found. The concepts are the same; the details differ.

| Topic | SQLite | PostgreSQL |
|---|---|---|
| Setup | nothing: one file, the module `sqlite3` | a server that runs in the background; here `pixeltable-pgserver` and `psycopg` |
| List tables and columns | `sqlite_master`, `pragma_table_info('t')` | `information_schema.tables`, `information_schema.columns` (standard SQL); `\dt`, `\d t` in `psql` |
| Dates | text `'YYYY-MM-DD'`; `julianday()`, `strftime()` | type `DATE`; `d1 - d2` in days, `EXTRACT`, `date_trunc`, `INTERVAL` |
| An impossible date such as `'2009-13-01'` | stored as text | rejected |
| True/false | the integers 1 and 0 | type `BOOLEAN`; `AVG(available::int)`, `COUNT(*) FILTER (WHERE ...)` |
| Text in an `INTEGER` column | stored, unless the table is `STRICT` | rejected |
| `CAST(160.71 AS INTEGER)` | 160 (cuts off) | 161 (rounds); `TRUNC` cuts off |
| `ROUND(x, 2)` on a float | works | needs `NUMERIC`: `ROUND(x::numeric, 2)` |
| Foreign keys | checked only after `PRAGMA foreign_keys = ON` | always checked |
| Bulk load | `executemany`, `to_sql`, `.import` in the `sqlite3` program | `COPY`, `\copy` in `psql` |
| A `SELECT` alias in `WHERE` | accepted | error, as in the SQL standard |
| `LIKE` | ignores case for A to Z | case-sensitive; `ILIKE` ignores case, also for `ö` |
| Escape character in `LIKE` | none unless `ESCAPE` is given | backslash by default |
| Regular expressions | `REGEXP` only with a function you register | `~`, `~*`, `!~`; word boundary `\y` |
| Parameters from Python | `?` | `%s` (and `%%` for a literal `%`) |
| NULL in `ORDER BY` | first in ascending order | last in ascending order |
| Compare with NULL as a value | `a IS b` | `a IS NOT DISTINCT FROM b` |
| Size of the course database | about 107 MB | about 300 MB |

## Stopping the server

The server stops by itself when the kernel shuts down. To stop it now, close the connection and call `server.cleanup()`, which stops the server when no other Python program uses it. The data stay in `case-study/data/airbnb/pgdata/`; the Setup cells start the server again. To delete the database completely, stop the server and delete that folder.

In [ ]:
con.close()
server.cleanup()        # stop the server; the data stay in the folder